<a href="https://colab.research.google.com/github/yewonhn/My-AI-Lab/blob/main/4%EC%A3%BC%EC%B0%A8/04_titanic_Classfication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 데이터 전처리
####  -과정
1. 라벨인코딩
2. 널 값 개수 확인
3. 결측치 확인 및 처리
4. 피처 셀렉션
5. 데이터 분리 및 준비

##0. 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/titanic.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [ ]:
X = df.drop('Survived',axis=1)
X.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
y = df['Survived']
y.head()

,Survived
0,0
1,1
2,1
3,1
4,0


In [ ]:
X = df.drop(['Survived', 'Name', 'Ticket', 'PassengerId', 'Cabin'], axis=1)
y = df['Survived']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier

##1. 널값 개수 확인
:데이터 셋에서 값이 비어있는 칸(결측치)가 몇개나 있는지 파악

In [ ]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


## 2. 라벨 인코딩


: 문자를 숫자로 바꿈
PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked 중에서 남길 것을 선별. (전처리 과정)

승객번호, Cabin(객실번호), 이름, 티켓은 문자여서 삭제 처리

## 3. 결측치 처리

확인된 결측치를 어떻게 할지 결정 - 값이 너무 없는 것은 지우고, 필요한 값인데 비어있다면 평균치를 구하여 넣음
(cabin은 지울거, age는 필요한데 177개가 null값. 평균값을 넣어주면 됨)

In [ ]:
# 1. 확실히 불필요한 컬럼 삭제
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# 2. 결측치 처리 (나이는 평균으로, 승선항은 최빈값으로)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 3. 남은 문자열(Sex, Embarked)을 숫자로 변환
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked']) # Embarked를 숫자로 자동 변환

print(df.head()) # 이제 모든 데이터가 숫자 형태가 됨

   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked_C  Embarked_Q  \
0         0       3    0  22.0      1      0   7.2500       False       False   
1         1       1    1  38.0      1      0  71.2833        True       False   
2         1       3    1  26.0      0      0   7.9250       False       False   
3         1       1    1  35.0      1      0  53.1000       False       False   
4         0       3    0  35.0      0      0   8.0500       False       False   

   Embarked_S  
0        True  
1       False  
2        True  
3        True  
4        True  


## 4. Feature Selection

: 좋은 재료만 골라내기
타겟(=생존여부)를 맞히는 데 방해가 되거나 상관없는 특징들은 제거함

불필요한 컬럼들 제거했으니, 학습에 사용할 X와 맞힐 정답 Y를 분리. 더 정교하게 하고 싶다면 상관계수를 확인.

In [ ]:
# 1. 정답(Target)인 Survived만 y로 추출
y = df['Survived']

# 2. Survived를 제외한 모든 데이터를 X(피처)로 설정
X = df.drop('Survived', axis=1)

# 3. 결과 확인
print("--- 피처(X) 목록 ---")
print(X.columns.tolist())
print("\n--- 데이터 샘플 (X.head()) ---")
print(X.head())

### 라벨의 개수 확인(수치)

ML_classfication_breast (결측치 없어도 결측치 제거하는 과정 등등 참고)

각 피처간의 관계를 보는 것이랑

sns.pairplot(df. vars = '~~

In [ ]:
# 정답(라벨)인 Survived 열의 데이터 개수 확인
print("--- 라벨 개수 확인 (수치) ---")
print(df['Survived'].value_counts())

# 비율로 확인하고 싶을 때 (normalize=True)
print("\n--- 라벨 비율 확인 (%) ---")
print(df['Survived'].value_counts(normalize=True) * 100)

### 훈련 데이터와 테스트 데이터

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X,
                                                  y,
                                                  test_size=0.2,    # 테스트는 20% 분량 준다는 것.
                                                  shuffle=True,
                                                  random_state=12)
print(X_train.shape, y_train.shape)  # 훈련 데이터 수
print(X_test.shape, y_test.shape)    # 테스트 데이터 수

(712, 11) (712,)
(179, 11) (179,)


###혼동행렬 출력
(생존 예측 생존)(사망 예측 생존)

(생존 예측 사망)(사망 예측 사망)

In [ ]:
clf_lr = LogisticRegression(random_state=0)

# Drop non-numeric columns from X_train and X_test before fitting
# Based on the kernel state, X_train still contains 'PassengerId', 'Name', 'Ticket', 'Sex', 'Cabin', 'Embarked'
problematic_cols = ['PassengerId', 'Name', 'Ticket', 'Sex', 'Cabin', 'Embarked']
# Use .copy() to avoid SettingWithCopyWarning if these are views
X_train_cleaned = X_train.drop(columns=problematic_cols, errors='ignore').copy()
X_test_cleaned = X_test.drop(columns=problematic_cols, errors='ignore').copy()

# Handle potential NaN values by filling with the mean (a simple imputation strategy)
X_train_cleaned = X_train_cleaned.fillna(X_train_cleaned.mean(numeric_only=True))
X_test_cleaned = X_test_cleaned.fillna(X_test_cleaned.mean(numeric_only=True))

clf_lr.fit(X_train_cleaned, y_train)

pred_lr = clf_lr.predict(X_test_cleaned)

print ("\n--- Logistic Regression Classifier ---")
print (accuracy_score(y_test, pred_lr))
print (confusion_matrix(y_test, pred_lr))


--- Logistic Regression Classifier ---
0.6536312849162011
[[86 14]
 [48 31]]
